In [ ]:
import torch
from torch import nn
from torch import optim

from torch.utils.data import DataLoader
from torchvision import datasets,transforms
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
#data _____-------------------------

transfrom = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))])
train_data = datasets.MNIST(root="./data", train=True, download=True, transform=transfrom)
test_data = datasets.MNIST(root="./data", train=False, download=True, transform=transfrom)
train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader = DataLoader(test_data, batch_size=64, shuffle=False)
#model _____________________________
class RNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.rnn = nn.RNN(input_size=28, hidden_size=128, num_layers=2, batch_first=True, nonlinearity="tanh")
        self.fc = nn.Linear(128, 10)

    def forward(self,x):
        x = x.squeeze(1)  # Remove the channel dimension
        out, hidden = self.rnn(x)
        out = out[:,-1,:]
        out = self.fc(out)
        return out
model = RNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr = 0.001)
print(f"paramter :,{sum(p.numel() for p in model.parameters()):}")
#train _________

epochs = 10 
train_losses = []
train_accuracies = []
print("\n" + "="*55)
print(f"{'Epoch':<8} {'Loss':>10} {'Accuracy':>12}")
print("="*55)

for epoch in range(epochs):
    model.train()
    total_loss = correct = total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        outputs        = model(images)
        loss           = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        predicted   = torch.argmax(outputs, dim=1)
        correct    += (predicted == labels).sum().item()
        total      += labels.size(0)

    avg_loss = total_loss / len(train_loader)
    accuracy = correct / total * 100
    train_losses.append(avg_loss)
    train_accuracies.append(accuracy)
    print(f"{epoch+1:<8} {avg_loss:>10.4f} {accuracy:>11.2f}%")

print("="*55)

model.eval()
correct = total = 0
with torch.no_grad():
    for images,labels in test_loader:
        images,labels = images.to(device),labels.to(device)
        predicted       = torch.argmax(model(images), dim=1)
        correct        += (predicted == labels).sum().item()
        total          += labels.size(0)

test_acc = correct / total * 100
print(f"\nTest Accuracy: {test_acc:.2f}%")

# ---- SAVE ----
torch.save(model.state_dict(), 'mnist_rnn.pth')
print("Model saved! ✅")
